In [ ]:
from pathlib import Path
import json
import pandas as pd

model_name = 'gemma12b'
json_path = Path(f"/nethome/soyoung/dynET/binding-iclr/results/path_patching/{model_name}/bid/binding_id_qk_intervention_group_b_head_from_logit_drop.json")


with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

def get_condition(data, key):
    if key in data.get("results", {}):
        return data["results"][key]
    if key in data.get("control_results", {}):
        return data["control_results"][key]
    raise KeyError(
        f"Condition {key!r} not found.\n"
        f"results keys: {list(data.get('results', {}).keys())}\n"
        f"control_results keys: {list(data.get('control_results', {}).keys())}"
    )

condition_order = [
    ("No intervention", "no_intervention"),
    ("Random Q", "random_query_shift"),
    ("Random K", "random_source_swap"),
    ("Random Q/K", "random_both"),
    ("Q shift", "query_shift"),
    ("K shift", "source_swap"),
    ("Q+K", "both_restore"),
    # ('Query box source shift', "question_query_box_source_shift"),
]

def get_nested(d, path, default=None):
    cur = d
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur

rows = []
for display_name, condition_key in condition_order:
    r = get_condition(data, condition_key)

    delta_by_entity = get_nested(r, ["final_logits", "mean_delta_logit_by_entity"], {}) or {}
    mean_object_delta = sum(delta_by_entity[str(i)] for i in range(3)) / 3
    target_delta = get_nested(r, ["final_logits", 'raw_delta_roles', 'target_answer'])

    rows.append({
        "Condition": display_name,
        "JSON key": condition_key,
        r"$\Delta R$": get_nested(
            r,
            ["attention", "overall", "log_ratio_target_over_original", "delta_after_minus_before"],
        ),
        r"$\Delta$ logit": get_nested(
            r,
            ["final_logits", "pairwise_delta_target_minus_original", "delta_after_minus_before"],
        ),
        "Raw target logit": get_nested(
            r,
            ["final_logits", 'raw_delta_roles', 'target_answer']
        ),
        "Switch frac.": get_nested(
            r,
            ["attention", "overall", "switch_original_to_target_fraction"],
        ),
        "Top-target frac.": get_nested(
            r,
            ["attention", "overall", "top_after_target_fraction"],
        ),
        "Raw original logit": get_nested(
            r,
            ["final_logits", 'raw_delta_roles', 'original_answer']
        ),
        "Centered target logit": target_delta - mean_object_delta,
    })

df = pd.DataFrame(rows)
df